# 01 · Data Acquisition

This project uses a **real, publicly-licensed credit-card transaction dataset** rather than
purely synthetic data: the *Credit Card Transactions Fraud Detection Dataset*
(originally published on Kaggle as `kartik2112/fraud-detection`, CC0 1.0 license),
generated with the open-source **Sparkov Data Generation** tool by simulating ~1,000
customer profiles transacting against ~700 real-world-style merchants over Jan 2019 -
Dec 2020, with fraud transactions injected by the same tool.

**Why this dataset over the alternatives:**

| Dataset | Problem |
|---|---|
| ULB `creditcard.csv` (the ubiquitous Kaggle "Credit Card Fraud Detection" set) | Features are PCA components `V1`...`V28` — anonymized into meaninglessness. No feature engineering or SHAP story is possible beyond "V14 matters." |
| IEEE-CIS Fraud Detection | Richer, but requires joining a Kaggle *competition* (auth + terms acceptance) and ships hundreds of opaque `V`/`C`/`D` engineered columns already. |
| **This dataset (Sparkov / kartik2112)** | Raw, interpretable fields: timestamp, merchant, category, amount, and — critically — **real latitude/longitude for both the cardholder and the merchant on every transaction**, which is exactly the raw material real feature engineering (velocity, geo-distance, merchant risk) is built from. |

Downloaded from a public, unauthenticated Hugging Face mirror of the same CSV, so this
notebook runs end-to-end without a Kaggle account.

Source: https://huggingface.co/datasets/dazzle-nu/CIS435-CreditCardFraudDetection

In [1]:
import os
import urllib.request
import pandas as pd
import numpy as np

RAW_CSV_URL = (
    "https://huggingface.co/datasets/dazzle-nu/CIS435-CreditCardFraudDetection"
    "/resolve/main/fraudTrain.csv"
)
RAW_CSV_PATH = "../data/raw/fraudTrain_source.csv"

os.makedirs("../data/raw", exist_ok=True)

if not os.path.exists(RAW_CSV_PATH):
    print(f"Downloading {RAW_CSV_URL} ...")
    urllib.request.urlretrieve(RAW_CSV_URL, RAW_CSV_PATH)
    print("Done.")
else:
    print(f"Already downloaded: {RAW_CSV_PATH} ({os.path.getsize(RAW_CSV_PATH) / 1e6:.1f} MB)")

Already downloaded: ../data/raw/fraudTrain_source.csv (266.5 MB)


## Load and clean

The community CSV mirror carries a couple of harmless ETL artifacts from however it was originally exported (a leading unnamed index column, occasional stray empty trailing columns) — dropped here. Nothing about the actual transaction data is altered.

In [2]:
raw = pd.read_csv(RAW_CSV_PATH, index_col=0, low_memory=False)

# Drop any fully-empty / unnamed artifact columns from the export
junk_cols = [c for c in raw.columns if raw[c].isna().all() or str(c).strip() == ""]
if junk_cols:
    print(f"Dropping empty artifact columns: {junk_cols}")
    raw = raw.drop(columns=junk_cols)

print(raw.shape)
raw.head()

Dropping empty artifact columns: ['Unnamed: 23', '6006']
(1048575, 22)


,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,1/1/19 0:00,2.703190e+15,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,...,36.0788,-81.1781,3495,"Psychologist, counselling",3/9/88,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1/1/19 0:00,6.304230e+11,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,...,48.8878,-118.2105,149,Special educational needs teacher,6/21/78,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,1/1/19 0:00,3.885950e+13,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,...,42.1808,-112.2620,4154,Nature conservation officer,1/19/62,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,1/1/19 0:01,3.534090e+15,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,...,46.2306,-112.1138,1939,Patent attorney,1/12/67,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,1/1/19 0:03,3.755340e+14,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,...,38.4207,-79.4629,99,Dance movement psychotherapist,3/28/86,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [3]:
raw.dtypes

trans_date_trans_time     object
cc_num                   float64
merchant                  object
category                  object
amt                      float64
first                     object
last                      object
gender                    object
street                    object
city                      object
state                     object
zip                        int64
lat                      float64
long                     float64
city_pop                   int64
job                       object
dob                       object
trans_num                 object
unix_time                  int64
merch_lat                float64
merch_long               float64
is_fraud                   int64
dtype: object

## Rename to a unified schema

Renamed to the field names the rest of this pipeline (feature engineering, model training, monitoring) expects — this is the only place those names are defined.

In [4]:
df = raw.rename(columns={
    "trans_date_trans_time": "timestamp",
    "cc_num": "user_id",
    "merchant": "merchant_id",
    "amt": "amount",
    "trans_num": "transaction_id",
    "lat": "home_lat",
    "long": "home_lon",
    "merch_lat": "merchant_lat",
    "merch_long": "merchant_lon",
})

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["dob"] = pd.to_datetime(df["dob"])
df["user_id"] = df["user_id"].astype(str)
df["merchant_id"] = df["merchant_id"].astype(str).str.replace("^fraud_", "", regex=True)
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"{len(df):,} transactions | {df.user_id.nunique():,} cards | {df.merchant_id.nunique():,} merchants")
print(f"Date range: {df.timestamp.min()} -> {df.timestamp.max()}")
print(f"Fraud rate: {df.is_fraud.mean():.4%}  ({df.is_fraud.sum():,} fraud transactions)")

/var/folders/82/g56w2bw94xq30h2kr2g3npm40000gn/T/ipykernel_24152/3179391534.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["timestamp"] = pd.to_datetime(df["timestamp"])


/var/folders/82/g56w2bw94xq30h2kr2g3npm40000gn/T/ipykernel_24152/3179391534.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["dob"] = pd.to_datetime(df["dob"])


1,048,575 transactions | 943 cards | 693 merchants
Date range: 2019-01-01 00:00:00 -> 2020-03-10 16:08:00
Fraud rate: 0.5728%  (6,006 fraud transactions)


## A note on responsible feature use

The raw data includes cardholder demographic fields — `gender`, `job`, `dob`, `city_pop`
(home-area population). We **deliberately exclude all of these from the model's feature
set** later in `03_feature_engineering.ipynb`: using demographic or socioeconomic proxies
in a fraud/credit-risk model creates real disparate-impact and fair-lending exposure, and a
production fraud team would need a documented governance review before ever using them.
We keep them in this raw table only for descriptive EDA (are they *correlated* with the
label at a population level? worth knowing), never as inputs the model sees.

## Geography sanity check

Confirms `home_lat`/`home_lon` are a fixed property of the cardholder (their registered address), not something that varies transaction-to-transaction — important, since our engineered "distance from home" feature later assumes this.

In [5]:
home_nunique = df.groupby("user_id")[["home_lat", "home_lon"]].nunique()
print("Cards with more than one distinct home location on record:", (home_nunique > 1).any(axis=1).sum(), "/", len(home_nunique))

# merchant location, by contrast, is simulated per-transaction (near the cardholder) rather
# than a fixed merchant coordinate -- check the spread for a single merchant:
sample_merchant = df.merchant_id.value_counts().index[0]
sample = df[df.merchant_id == sample_merchant][["merchant_lat", "merchant_lon"]]
print(f"\nmerchant '{sample_merchant}' appears {len(sample)} times with "
      f"{sample.merchant_lat.nunique()} distinct lat values -- confirms merchant lat/lon is "
      f"simulated per-transaction (near the cardholder), not a fixed storefront location.")
print("Implication: a 'distance from home' feature is well-grounded here; chaining "
      "consecutive transactions into a literal travel-speed/'impossible travel' feature "
      "would not be, since merchant coordinates aren't a continuous real path. We use the "
      "former and skip the latter in 03_feature_engineering.ipynb.")

Cards with more than one distinct home location on record: 21 / 943



merchant 'Kilback LLC' appears 3521 times with 3521 distinct lat values -- confirms merchant lat/lon is simulated per-transaction (near the cardholder), not a fixed storefront location.
Implication: a 'distance from home' feature is well-grounded here; chaining consecutive transactions into a literal travel-speed/'impossible travel' feature would not be, since merchant coordinates aren't a continuous real path. We use the former and skip the latter in 03_feature_engineering.ipynb.


## Reference tables

A `users` table (one row per card: home location + demographics, kept for EDA only) and a `merchants` table (one row per merchant: dominant category + average simulated location), analogous to the reference tables a real feature store would maintain.

In [6]:
users = (
    df.sort_values("timestamp")
      .groupby("user_id")
      .first()[["home_lat", "home_lon", "gender", "job", "dob", "city", "state", "zip", "city_pop"]]
      .reset_index()
)
users["first_seen"] = df.groupby("user_id")["timestamp"].min().values

merchants = (
    df.groupby("merchant_id")
      .agg(category=("category", lambda s: s.mode().iat[0]),
           avg_lat=("merchant_lat", "mean"),
           avg_lon=("merchant_lon", "mean"),
           n_txns=("merchant_id", "size"),
           fraud_rate=("is_fraud", "mean"))
      .reset_index()
)

print(f"users: {users.shape}, merchants: {merchants.shape}")
merchants.sort_values("fraud_rate", ascending=False).head(10)

users: (943, 11), merchants: (693, 6)


,merchant_id,category,avg_lat,avg_lon,n_txns,fraud_rate
245,"Herman, Treutel and Dickens",misc_net,38.448716,-90.839501,1026,0.027290
337,Kozey-Boehm,shopping_net,38.526768,-90.465871,1513,0.025116
346,Kuhic LLC,shopping_net,38.447171,-90.239438,1593,0.021971
72,Boyer-Reichert,shopping_net,38.536797,-90.065426,1547,0.021332
200,Goyette Inc,shopping_net,38.600320,-90.675380,1599,0.021263
616,Terry-Huel,shopping_net,38.320562,-90.433814,1606,0.021171
175,Fisher-Schowalter,shopping_net,38.521072,-89.747068,1574,0.020966
630,"Towne, Greenholt and Koepp",shopping_net,38.423482,-90.355015,1575,0.020952
282,Jast Ltd,shopping_net,38.379897,-90.677893,1585,0.020820
193,Gleason-Macejkovic,shopping_net,38.529833,-90.865541,1647,0.020644


## Save

In [7]:
keep_cols = ["transaction_id", "timestamp", "user_id", "amount", "merchant_id", "category",
             "home_lat", "home_lon", "merchant_lat", "merchant_lon", "city", "state", "is_fraud"]
transactions = df[keep_cols].copy()

transactions.to_parquet("../data/raw/transactions.parquet", index=False)
users.to_parquet("../data/raw/users.parquet", index=False)
merchants.to_parquet("../data/raw/merchants.parquet", index=False)

print("Saved:")
print(f"  data/raw/transactions.parquet  {transactions.shape}")
print(f"  data/raw/users.parquet         {users.shape}")
print(f"  data/raw/merchants.parquet     {merchants.shape}")

Saved:
  data/raw/transactions.parquet  (1048575, 13)
  data/raw/users.parquet         (943, 11)
  data/raw/merchants.parquet     (693, 6)
